Goals: 
establish the following
* lap boundaries: the times when the car crosses the start/finish line
* lap count and lap times
* lap status: which laps are full timed laps, and which are out-laps (starting from standstill or the pit), in-laps (ending in the pit), or contain a common gap

* s goes 0 -> 805 and wraps back to 0 at the line
* so a lap boundary is where s suddenly drops by a large amount (from ~800 to ~0)
* lap time = the time between two consecutive boundaries

one of the issues we discovered in /notebooks/02_session.ipynb is:
- lap distance s flickers between ~805m and 0 when the car stops at the start/finish line (~975-980s), that produces lap reset so must be handled.


In [12]:
from src.loader import load
df = load("vehicle_curvilinear_coordinates")[["t_s", "s"]]

mask = (df["t_s"] >= 971) & (df["t_s"] <= 977)
mask2 = (df["s"] < 700)
flicker = df[mask & mask2]
segment = df[mask]

In [8]:
diff = flicker["s"].max() - flicker["s"].min()
diff

np.float64(6.2821958790210175)

In [13]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


fig = make_subplots(rows=1, cols=1, shared_xaxes=True)

fig.add_trace(
    go.Scattergl(
    x = segment["t_s"],
    y = segment["s"],
    name="s"
),
    row =1,
    col =1)

fig.update_yaxes(title_text="s", row=1, col=1)
fig.update_layout(
    height=300,
    width = 800,
    showlegend=True
)

fig.show()

hence, we need to think of what's physically true about a real line crossing that isn't true of the flicker

physically, we can follow two "rules":
1. a lap boundary is where s drops by more than half the track length AND the car is moving (wheel speed > motion_threshold_rad_s)
2. laps can't be impossibly short. min lap time = track length / the session's max speed

**assumption**: the car crosses the line driving forward. 
